| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 3: Data Preparation | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.2 | 13-12-2025 | Júlio César e Lays de Freitas | Concluído |

Esse Notebook contém o **pré-processamento dos dados**.

### BIBLIOTECAS

In [35]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from datetime import date

### CARREGAMENTO DOS DADOS

In [36]:
# Listar os arquivos CSV na pasta 'uploads'
arquivos_csv = glob.glob('uploads/processos_*.csv')

# Carregar os arquivos CSV e concatenar em um único DataFrame
dfs = []
for arquivo in arquivos_csv:
    
    df_temp = pd.read_csv(arquivo, sep=',', encoding='utf-8')
    dfs.append(df_temp)

dataset = pd.concat(dfs, ignore_index=True)

print("\n=== Arquivo carregado com sucesso! ===")
print("Dimensões (linhas, colunas):", dataset.shape)


=== Arquivo carregado com sucesso! ===
Dimensões (linhas, colunas): (3245632, 10)


### GRAVANDO UMA CÓPIA PARA TRABALHO

In [37]:
df = dataset.copy()

### AMOSTRA DOS DADOS

In [38]:
df.head()

,processo,data_distribuicao,data_baixa,entrancia,comarca,serventia,nome_area_acao,is_segredo_justica,codg_classe,codg_assuntos
0,0119071.75.2004.8.09.0051,2022-05-25,2022-06-30,FINAL,GOIÂNIA,2ª Vara Cível,upj civel,False,7.0,10671
1,0168391.94.2004.8.09.0051,2022-05-20,2022-05-20,FINAL,GOIÂNIA,3ª Vara Cível,upj civel,False,7.0,10671
2,0189657.40.2004.8.09.0051,2022-06-02,2024-01-22,FINAL,GOIÂNIA,31ª Vara Cível,upj civel,False,7.0,10671
3,0197944.89.2004.8.09.0051,2022-06-07,2022-10-07,FINAL,GOIÂNIA,22ª Vara Cível,upj civel,False,7.0,10671
4,0211274.56.2004.8.09.0051,2022-06-09,2022-08-03,FINAL,GOIÂNIA,8ª Vara Cível,upj civel,False,7.0,10671


In [53]:
df_temp = df.copy()
df_temp['data_baixa'] = df_temp['data_baixa'].astype(str)
df_temp[(df_temp['comarca'] == 'VARJÃO') & (df_temp['serventia'] == 'Vara Judicial') & (df_temp['data_baixa'].str.contains('2022-06'))]

,processo,data_distribuicao,data_baixa,entrancia,comarca,serventia,nome_area_acao,is_segredo_justica,codg_classe,codg_assuntos,ano_distribuicao,mes_distribuicao,dia_distribuicao,ano_baixa,mes_baixa,dia_baixa
45155,5046834.98.2022.8.09.0156,2022-01-29,2022-06-27,INICIAL,VARJÃO,Vara Judicial,familia - interior,False,12372.0,6239,2022,1,29,2022.0,6.0,27.0
47974,5049765.94.2022.8.09.0117,2022-01-31,2022-06-03,INICIAL,VARJÃO,Vara Judicial,juizado especial civel,False,241.0,11974,2022,1,31,2022.0,6.0,3.0
56509,5058673.43.2022.8.09.0117,2022-02-04,2022-06-30,INICIAL,VARJÃO,Vara Judicial,criminal,False,279.0,3608,2022,2,4,2022.0,6.0,30.0
73860,5077050.62.2022.8.09.0117,2022-02-14,2022-06-21,INICIAL,VARJÃO,Vara Judicial,infancia e juventude infracional,True,1463.0,9860,2022,2,14,2022.0,6.0,21.0
134193,5140448.80.2022.8.09.0117,2022-03-14,2022-06-20,INICIAL,VARJÃO,Vara Judicial,criminal,False,355.0,15036,2022,3,14,2022.0,6.0,20.0
137716,5144161.43.2022.8.09.0156,2022-03-15,2022-06-07,INICIAL,VARJÃO,Vara Judicial,familia - interior,True,69.0,6239,2022,3,15,2022.0,6.0,7.0
157810,5165116.95.2022.8.09.0156,2022-03-23,2022-06-06,INICIAL,VARJÃO,Vara Judicial,familia - interior,True,69.0,6239,2022,3,23,2022.0,6.0,6.0
163564,5171787.57.2022.8.09.0117,2022-03-25,2022-06-07,INICIAL,VARJÃO,Vara Judicial,familia - interior,False,12247.0,10938,2022,3,25,2022.0,6.0,7.0
164287,5172531.52.2022.8.09.0117,2022-03-25,2022-06-06,INICIAL,VARJÃO,Vara Judicial,familia - interior,True,69.0,6239,2022,3,25,2022.0,6.0,6.0
164366,5172616.38.2022.8.09.0117,2022-03-25,2022-06-03,INICIAL,VARJÃO,Vara Judicial,familia - interior,True,69.0,6239,2022,3,25,2022.0,6.0,3.0


### LIMPEZA E TRATAMENTO DOS DADOS

In [39]:
# Verificar o nome correto das colunas (pode haver diferenças de acentuação ou espaços)
colunas = df.columns.tolist()

# Encontrar as colunas de data corretamente
coluna_serventia = [col for col in colunas if 'serventia' in col.lower()][0]
coluna_distribuicao = [col for col in colunas if 'data_distribuicao' in col.lower()][0]
coluna_baixa = [col for col in colunas if 'data_baixa' in col.lower()][0]
coluna_area_acao = [col for col in colunas if 'nome_area_acao' in col.lower()][0]
coluna_processo_id = [col for col in colunas if 'processo' in col.lower()][0]
coluna_comarca = [col for col in colunas if 'comarca' in col.lower()][0]

# Renomear colunas para garantir consistência
df = df.rename(columns={
coluna_distribuicao: 'data_distribuicao',
coluna_baixa: 'data_baixa',
coluna_area_acao: 'nome_area_acao',
coluna_processo_id: 'processo',
coluna_comarca: 'comarca',
coluna_serventia: 'serventia'
})

# Converter colunas de data para datetime com tratamento de erros
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

### CONSTRUÇÃO DO DATAFRAME DE TREINO E TESTE

In [40]:
# CRIAÇÃO DAS ESTATÍSTICAS POR MÊS ('comarca' e 'serventia') >> Revisado
# --- 1. PREPARAÇÃO DOS DADOS ---
# Extração de componentes de data
print("Extraindo datas...")
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['mes_distribuicao'] = df['data_distribuicao'].dt.month
df['dia_distribuicao'] = df['data_distribuicao'].dt.day

df['ano_baixa'] = df['data_baixa'].dt.year
df['mes_baixa'] = df['data_baixa'].dt.month
df['dia_baixa'] = df['data_baixa'].dt.day

# Chaves de agrupamento
grouping_keys = ['comarca', 'serventia']

# ==============================================================================
# FUNÇÃO GENÉRICA DE CÁLCULO (Para evitar repetição de código)
# ==============================================================================
def calcular_estatisticas_cohort(df_main, cols_dist, cols_baixa, nome_periodo):
    """
    Calcula Distribuídos, Baixados e Pendentes.
    REVISÃO DOS CÁLCULOS:
      - Distribuídos: contagem por data_distribuicao (entrada)
      - Baixados: contagem por data_baixa (referência), dentro do par entrada->referência
      - Pendentes: contagem de data_baixa nula/vazia (NaT), agrupada por entrada
    """
    
    # 1. Calcular TOTAL DE DISTRIBUÍDOS
    cols_group_dist = cols_dist + grouping_keys
    df_dist = df_main.groupby(cols_group_dist)['processo'].nunique().reset_index(name=f'Distribuídos{nome_periodo}')
    
    # 2. Calcular TOTAL DE BAIXADOS (somente registros com baixa)
    cols_group_baixa = cols_dist + cols_baixa + grouping_keys
    df_baixa = df_main.dropna(subset=cols_baixa).groupby(cols_group_baixa)['processo'].nunique().reset_index(name=f'Baixados{nome_periodo}')
    
    # 3. Calcular TOTAL DE PENDENTES (data_baixa nula/vazia -> componentes de baixa NaN)
    df_pend = df_main[df_main[cols_baixa[0]].isna()].groupby(cols_group_dist)['processo'].nunique().reset_index(name=f'Pendentes{nome_periodo}')
    
    # 4. CRIAÇÃO DO GRID (Cross Join)
    unique_dist = df_dist[cols_dist].drop_duplicates()
    unique_baixa = df_main[cols_baixa].dropna().drop_duplicates()
    unique_units = df_main[grouping_keys].drop_duplicates()
    
    # Cross Join 1: Datas de Dist x Datas de Baixa (usando merge dummy para performance)
    df_dates = pd.merge(
        unique_dist.assign(key=1), 
        unique_baixa.assign(key=1), 
        on='key'
    ).drop('key', axis=1)
    
    # --- FILTRO DE DATAS ---
    if len(cols_dist) == 1: # Anual
        df_dates = df_dates[df_dates[cols_baixa[0]] >= df_dates[cols_dist[0]]]
        
    elif len(cols_dist) == 2: # Mensal
        ano_d = df_dates[cols_dist[0]].astype(int).astype(str)
        mes_d = df_dates[cols_dist[1]].astype(int).astype(str)
        
        ano_b = df_dates[cols_baixa[0]].astype(int).astype(str)
        mes_b = df_dates[cols_baixa[1]].astype(int).astype(str)
        
        d_dist = pd.to_datetime(ano_d + '-' + mes_d + '-01')
        d_baixa = pd.to_datetime(ano_b + '-' + mes_b + '-01')
        
        df_dates = df_dates[d_baixa >= d_dist]

    # Cross Join 2: (Datas) x (Comarca/Serventia)
    df_grid = pd.merge(
        df_dates.assign(key=1),
        unique_units.assign(key=1),
        on='key'
    ).drop('key', axis=1)
    
    # 5. MERGES (Juntar dados reais no Grid)
    df_final = pd.merge(df_grid, df_dist, on=cols_dist + grouping_keys, how='left')
    df_final = pd.merge(df_final, df_baixa, on=cols_dist + cols_baixa + grouping_keys, how='left')
    df_final = pd.merge(df_final, df_pend, on=cols_dist + grouping_keys, how='left')
    
    # Preencher Zeros
    df_final[f'Distribuídos{nome_periodo}'] = df_final[f'Distribuídos{nome_periodo}'].fillna(0).astype(int)
    df_final[f'Baixados{nome_periodo}'] = df_final[f'Baixados{nome_periodo}'].fillna(0).astype(int)
    df_final[f'Pendentes{nome_periodo}'] = df_final[f'Pendentes{nome_periodo}'].fillna(0).astype(int)
    
    # Filtrar apenas onde houve distribuição
    df_final = df_final[df_final[f'Distribuídos{nome_periodo}'] > 0].copy()
    
    # 6. TAXA DE CONGESTIONAMENTO (com a definição solicitada)
    soma = df_final[f'Baixados{nome_periodo}'] + df_final[f'Pendentes{nome_periodo}']
    df_final[f'Taxa de Congestionamento{nome_periodo} (%)'] = np.where(
        soma > 0, (df_final[f'Pendentes{nome_periodo}'] / soma) * 100, 0
    ).round(2)

    # 7. CONVERSÃO FINAL PARA INTEIRO (NOVO BLOCO)
    cols_tempo = cols_dist + cols_baixa
    for col in cols_tempo:
        if col in df_final.columns:
            df_final[col] = df_final[col].astype(int)

    return df_final

# ==============================================================================
# 2. MONTANDO O DATASET COM OS CÁLCULOS MENSAIS
# ==============================================================================
print("Calculando estatísticas mensais...")
df_estatisticas = calcular_estatisticas_cohort(
    df, 
    cols_dist=['ano_distribuicao', 'mes_distribuicao'], 
    cols_baixa=['ano_baixa', 'mes_baixa'], 
    nome_periodo='_mes'
)

# Exclusão de Colunas Desnecessárias
cols_to_drop = [
    'ano_distribuicao',    
    'mes_distribuicao', 
    'ano_baixa',       
    'mes_baixa'                   
]

# Dropamos apenas o que existe no dataframe
df2 = df_estatisticas.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

df3 = df2.copy()

# Inclusão da coluna mês de referência para Aplicação de ML
df3["mes_ref"] = pd.to_datetime(
    df_estatisticas["ano_baixa"].astype(str) + "-" + df_estatisticas["mes_baixa"].astype(str).str.zfill(2) + "-01"
)

# Dataset pronto
df_estatisticas_mes = df3.sort_values("mes_ref")

print("Concluído!")

Extraindo datas...
Calculando estatísticas mensais...
Concluído!


### AMOSTRA DO DATAFRAME TRATADO

In [41]:
df_estatisticas_mes.head()

,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%),mes_ref
5662,LUZIÂNIA,1° Juizado Especial Cível e Criminal,416,1,15,93.75,2022-01-01
5802,MORRINHOS,Juizado Especial Cível e Criminal,106,0,0,0.00,2022-01-01
5803,TRIBUNAL DE JUSTIÇA,GABINETE DESA. ROBERTA NASSER LEONE,51,0,1,100.00,2022-01-01
5804,RIO VERDE,3ª Vara Cível,79,0,19,100.00,2022-01-01
5805,TRIBUNAL DE JUSTIÇA,GABINETE DES. FABIO CRISTOVAO DE CAMPOS FARIA,64,0,0,0.00,2022-01-01


In [42]:
df_estatisticas_mes.to_csv('datasets/df_estatisticas_mes.csv', index=False)

In [43]:
df_estatisticas_mes.sample()

,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%),mes_ref
331239,GOIÂNIA,2º Juizado Especial Cível,356,1,14,93.33,2025-06-01
